# 03 — Rede neural (Integrante 2)

Treina uma MLP (PyTorch) para estimar a glicose média (mg/dL) a partir das tabelas geradas pelo pipeline do Integrante 1 (`preprocessing.py`).

Usa `day_level` (mais linhas que `wide`, conforme sugestão do README). Os splits e a normalização já vêm prontos em `data/processed/model_ready/day_level/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import model as M
import preprocessing as P

TABLE = "day_level"  # alternativa: "wide"
DATA_DIR = ROOT / "data/processed/model_ready" / TABLE

## 1. Carregar dados já escalados

`X_*` já passou por `FittedPreprocessor` (imputação pela mediana + `StandardScaler`, ajustados só no treino).

In [ ]:
TARGET = "target_gl_mean" if TABLE == "day_level" else "target_d9_gl_mean"

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")[TARGET]
X_valid = pd.read_csv(DATA_DIR / "X_valid.csv")
y_valid = pd.read_csv(DATA_DIR / "y_valid.csv")[TARGET]
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv")[TARGET]

print(f"train={X_train.shape} valid={X_valid.shape} test={X_test.shape}")

## 2. Treinar a MLP

Early stopping na perda (MSE) de validação. Ajuste `hidden_sizes`, `dropout`, `weight_decay` aqui se o modelo estiver sobreajustando (poucas linhas: ~212 no treino de `day_level`).

In [ ]:
result = M.train_model(
    X_train, y_train, X_valid, y_valid,
    hidden_sizes=(64, 32),
    dropout=0.3,
    lr=1e-3,
    weight_decay=1e-3,
    max_epochs=500,
    patience=30,
)

plt.plot(result.history["train_loss"], label="treino")
plt.plot(result.history["valid_loss"], label="validação")
plt.axvline(result.best_epoch - 1, color="gray", linestyle="--", label="melhor época")
plt.xlabel("época"); plt.ylabel("MSE"); plt.legend(); plt.title("Curva de treino")
plt.show()

## 3. Avaliar (valid e test)

MAE/RMSE em mg/dL, R² na mesma escala do alvo. Compare com o baseline (prever a média do treino) para saber se a rede está de fato aprendendo algo.

In [ ]:
import numpy as np

baseline_pred = np.full(len(y_test), y_train.mean())
baseline_mae = np.abs(baseline_pred - y_test).mean()

print("valid:", M.evaluate(result.model, X_valid, y_valid))
print("test: ", M.evaluate(result.model, X_test, y_test))
print(f"baseline (prever média do treino) MAE no test: {baseline_mae:.2f}")

## 4. Salvar o modelo treinado

Salva pesos (`.pt`) + arquitetura (`.json` ao lado). `model.load_model(...)` recarrega os dois.

In [ ]:
MODEL_PATH = ROOT / "models" / f"{TABLE}_mlp.pt"
M.save_model(result.model, MODEL_PATH)
print(f"Salvo em {MODEL_PATH}")

## 5. Previsão em dados novos (interface para o Integrante 3)

O algoritmo genético gera "planos" com as colunas originais (não escaladas) e passa por `prep.transform(...)` antes da rede — é o mesmo caminho usado aqui.

In [ ]:
prep = P.load_preprocessor(DATA_DIR / "preprocessor.pkl")
reloaded = M.load_model(MODEL_PATH)

# exemplo: reconstruir um plano a partir de uma linha do treino, só pra validar o caminho ponta a ponta
# (dataset_day_level já tem as colunas prev_* que o preprocessor deste TABLE espera)
day_level = pd.read_csv(ROOT / "data/processed/dataset_day_level.csv")
exemplo = day_level[day_level["split"] == "input"].iloc[[0]]
pred = M.predict(reloaded, exemplo, prep, clip=5)
print("previsão (mg/dL):", pred)